# Using the PhenoMe pipeline with STED-FM models and data
## Zooniverse - Synaptic protein clusters segmentation
---

## 1. Setup

In [64]:
import torch
import numpy as np
from phenome import PhenoMe, get_metadata_from_path, load_dinov2_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

### Wrap the STED-FM model

In [65]:
from stedfm import get_pretrained_model_v2
from phenome.utils import ModelWrapper

class STEDFMWrapper(ModelWrapper):
    def __init__(self, device):
        # Load the pretrained STED-FM model
        model, cfg = get_pretrained_model_v2(
            name = "mae-lightning-small",
            weights = "MAE_SMALL_STED", 
            in_channels=1, # For single channel images
            as_classifier=True,
            # global_pool = "patch" # Returns the embeddings for all patches
        )

        # Initialize the parent class which moves the model to the device and sets it to eval()
        super().__init__(model=model, device=device)

    def _get_embeddings(self, tensor: torch.Tensor) -> torch.Tensor:
        out = self.model.forward_features(tensor)
        return out

wrapper = STEDFMWrapper(device=device)

in_channels 1
--- mae-lightning-small | (https://s3.valeria.science/flclab-foundation-models/models/mae-small-sted.zip) Downloading from URL and extracting zip ---

Loading state_dict from /Users/renaud/.stedfm/baselines/mae-small_STED/pl_checkpoint-999.pth
--- Loaded model mae-lightning-small with weights MAE_SMALL_STED ---
--- Added linear probe to all frozen blocks ---


### Using DINOv2

In [24]:
wrapper = load_dinov2_model(model_name="dinov2_vits14_reg", device=device)

Using cache found in /Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main
/Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


## Synaptic Proteins Dataset

### Zenodo segmentation dataset

In [45]:
from data_handling import SemanticProteinSegmentationDataset
import h5py
from matplotlib import pyplot as plt

h5file = "/Users/renaud/datasets/synaptic_proteins/test_2024-05-16.hdf5"

dataset = SemanticProteinSegmentationDataset(
    h5file=h5file,
    n_channels=1,
    data_aug=0,
    validation=True,
    image_size=1024,
    return_metadata=True,  # __getitem__ also yields the protein/condition of each tile
)


100%|██████████| 1357/1357 [00:00<00:00, 2998.17it/s]


In [71]:
from data_handling import export_cluster_crops

# From the dataset containing large crops, find each individual synaptic protein cluster and export their cropped image and mask

export_cluster_crops(
    dataset=dataset,
    output_dir="/Users/renaud/datasets/synaptic_proteins/clusters_segmentation_dist8_area256_minmax_dataset",
    min_distance=8,
    threshold=0.3,
    min_area=256,
    intensity_mode="image_percentile",
    bbox_padding=2,
    pad_to_size=None
)

,filename,image_name,channel,protein,partner_protein,antibody_label,condition,replicate,synapse_class,class_index,...,content_height,content_width,min_distance,bbox_padding,touches_image_border,cluster_id,class_majority,seg_majority,class_agreement,n_class_votes
0,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,0,Homer,PSD95,GARSTAR635,GluGly,01,round,0,...,33,30,8,2,False,321,0,4,0.4,10
1,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,0,Homer,PSD95,GARSTAR635,GluGly,01,round,0,...,53,39,8,2,False,352,0,8,0.4,10
2,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,0,Homer,PSD95,GARSTAR635,GluGly,01,round,0,...,24,25,8,2,False,394,0,3,0.5,10
3,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,0,Homer,PSD95,GARSTAR635,GluGly,01,round,0,...,30,23,8,2,False,492,0,5,0.4,10
4,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,0,Homer,PSD95,GARSTAR635,GluGly,01,round,0,...,26,38,8,2,False,636,0,5,0.4,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,1,Bassoon,Rim,GAMAlexa594,0MgGlyBic,02,multidomains,3,...,36,24,8,2,False,910,3,5,0.7,10
607,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,1,Bassoon,Rim,GAMAlexa594,0MgGlyBic,02,multidomains,3,...,47,31,8,2,False,961,3,3,0.8,10
608,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,1,Bassoon,Rim,GAMAlexa594,0MgGlyBic,02,multidomains,3,...,50,39,8,2,False,962,3,3,0.4,10
609,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,1,Bassoon,Rim,GAMAlexa594,0MgGlyBic,02,multidomains,3,...,29,30,8,2,False,993,3,5,0.5,10


### Custom transforms for STED-FM model

In [ ]:
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image

TARGET_SIZE = 224


class ResizeLongestSide:
    """Resize a PIL image so that its longest side matches `size`, keeping the aspect ratio.

    Args:
        size: Target length, in pixels, of the longest side.
    """

    def __init__(self, size: int = TARGET_SIZE) -> None:
        self.size = size

    def __call__(self, img: Image.Image) -> Image.Image:
        """Resize the image.

        Args:
            img: Input PIL image.

        Returns:
            Image.Image: Image whose longest side equals `size`.
        """
        width, height = img.size
        scale = self.size / max(width, height)
        return TF.resize(img, [max(1, round(height * scale)), max(1, round(width * scale))], interpolation=T.InterpolationMode.NEAREST)

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}(size={self.size})"


class AnchoredPad:
    """Pad a PIL image to a square `size` without rescaling it.

    Images larger than `size` along an axis are cropped with the same anchor, so the
    output is always `size` x `size`.

    Args:
        size: Side length, in pixels, of the square output.
        anchor: Placement of the original image, either "top_left" or "center".
        fill: Value used for the padded pixels.
    """

    def __init__(self, size: int = TARGET_SIZE, anchor: str = "top_left", fill: int = 0) -> None:
        if anchor not in ("top_left", "center"):
            raise ValueError(f"anchor must be 'top_left' or 'center', got {anchor!r}")
        self.size = size
        self.anchor = anchor
        self.fill = fill

    def __call__(self, img: Image.Image) -> Image.Image:
        """Pad (and crop if needed) the image to `size` x `size`.

        Args:
            img: Input PIL image.

        Returns:
            Image.Image: Square image of side `size`.
        """
        width, height = img.size
        pad_w, pad_h = max(0, self.size - width), max(0, self.size - height)
        left, top = (pad_w // 2, pad_h // 2) if self.anchor == "center" else (0, 0)
        img = TF.pad(img, [left, top, pad_w - left, pad_h - top], fill=self.fill)
        if self.anchor == "center":
            return TF.center_crop(img, [self.size, self.size])
        return TF.crop(img, 0, 0, self.size, self.size)

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}(size={self.size}, anchor={self.anchor!r})"


# Transforms for STED-FM model: Naive resize to 224x224; stretches the image to dimensions
stedfm_stretch_transforms = T.Compose([
    T.ToPILImage(),
    T.Resize((TARGET_SIZE, TARGET_SIZE), interpolation=T.InterpolationMode.NEAREST),
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),
])

# Keep the crop as is, pad to 224x224 with image in top left corner
stedfm_pad_transforms = T.Compose([
    T.ToPILImage(),
    AnchoredPad(TARGET_SIZE, anchor="top_left"),
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),
])

# Set the longest side of the image to 224, and pad the rest
stedfm_resize_and_pad_transforms = T.Compose([
    T.ToPILImage(),
    ResizeLongestSide(TARGET_SIZE),
    AnchoredPad(TARGET_SIZE, anchor="top_left"),
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),
])

# Set the shortest side of the image to 224, then center crop
stedfm_resize_and_crop_transforms = T.Compose([
    T.ToPILImage(),
    T.Resize(TARGET_SIZE, interpolation=T.InterpolationMode.NEAREST),  # int size resizes the shortest side, keeping the aspect ratio
    T.CenterCrop(TARGET_SIZE),
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),
])

# Transforms for DinoV2 model
dinov2_stretch_transforms = T.Compose([
    T.ToPILImage(),
    T.Resize((TARGET_SIZE, TARGET_SIZE), interpolation=T.InterpolationMode.NEAREST),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

# Transforms for DinoV2 model
dinov2_pad_transforms = T.Compose([
    T.ToPILImage(),
    AnchoredPad(TARGET_SIZE, anchor="top_left"),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

# Transforms for DinoV2 model
dinov2_resize_and_pad_transforms = T.Compose([
    T.ToPILImage(),
    ResizeLongestSide(TARGET_SIZE),
    AnchoredPad(TARGET_SIZE, anchor="top_left"),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

# Transforms for DinoV2 model
dinov2_resize_and_crop_transforms = T.Compose([
    T.ToPILImage(),
    T.Resize(TARGET_SIZE, interpolation=T.InterpolationMode.NEAREST),  # int size resizes the shortest side, keeping the aspect ratio
    T.CenterCrop(TARGET_SIZE),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

In [ ]:
import matplotlib.pyplot as plt
from skimage.io import imread

# Test custom transforms

test_img = torch.rand((1,40,80))

plt.imshow(stedfm_stretch_transforms(test_img)[0])
plt.show()

plt.imshow(stedfm_pad_transforms(test_img)[0])
plt.show()

plt.imshow(stedfm_resize_and_pad_transforms(test_img)[0])
plt.show()

plt.imshow(stedfm_resize_and_crop_transforms(test_img)[0])
plt.show()

### Create dataset (find files)

In [72]:
# csv-based metadata
from phenome import make_dataframe_metadata_fn
from data_handling import minmax_preprocessing_fn
import pandas as pd

images_dir = "/Users/renaud/datasets/synaptic_proteins/clusters_segmentation_dist8_area256_minmax_dataset/images"
masks_dir = "/Users/renaud/datasets/synaptic_proteins/clusters_segmentation_dist8_area256_minmax_dataset/masks"
metadata_dir = "/Users/renaud/datasets/synaptic_proteins/clusters_segmentation_dist8_area256_minmax_dataset/metadata.csv"

if metadata_dir:
    metadata_df = pd.read_csv(metadata_dir)
    metadata_fn = make_dataframe_metadata_fn(metadata_df)
else:
    metadata_fn = None
file_df = pheno.find_files(images_dir, mask_dir=masks_dir, metadata_fn=metadata_fn)

print(f"Found {len(file_df)} images")
print(f"Metadata keys: {file_df.columns.tolist()}")



Found 611 files total.
Matched masks: 611/611 images (0 missing).
DataFrame: 611 files, 33 columns.


Found 611 images
Metadata keys: ['file_path', 'filename', 'image_name', 'channel', 'protein', 'partner_protein', 'antibody_label', 'condition', 'replicate', 'synapse_class', 'class_index', 'centroid_row', 'centroid_col', 'bbox_row_min', 'bbox_col_min', 'bbox_row_max', 'bbox_col_max', 'area', 'n_components', 'crop_height', 'crop_width', 'content_height', 'content_width', 'min_distance', 'bbox_padding', 'touches_image_border', 'cluster_id', 'class_majority', 'seg_majority', 'class_agreement', 'n_class_votes', 'id', 'mask_path']


### Process images

In [74]:
# Process with STED-FM model

# Select transformations from:
#   stedfm_stretch_transforms
#   stedfm_pad_transforms
#   stedfm_resize_and_pad_transforms
#   stedfm_resize_and_crop_transforms
custom_transformations = stedfm_stretch_transforms
checkpoint_path = "checkpoints/segmentation_dataset_clusters/clusters_stretch_dist8_area256_stedfm.h5"

pheno.process_images(wrapper, 
                    force_rgb=False,
                    checkpoint_path=checkpoint_path,
                    custom_transformations=custom_transformations,
                    l2_normalize_channels=False
                    )
pheno.compute_properties(property_preset="complete", checkpoint_path=checkpoint_path)

Processing batches:   0%|          | 0/20 [00:00<?, ?it/s]

Pipeline ready: 611 images available (lazy loading from checkpoints/segmentation_dataset_clusters/clusters_stretch_dist8_area256_stedfm.h5).
Masks: 611/611 images.


Computing properties:   0%|          | 0/611 [00:00<?, ?img/s]

Checkpoint: saved 611 properties (refreshed).


,image_index,image_path,image_name,intensity_max,perimeter,eccentricity,intensity_max_masked,texture_correlation,intensity_std_masked,ring_3_std,...,major_axis_length,texture_homogeneity,orientation,ring_3_mean,solidity,equivalent_diameter,intensity_entropy,intensity_mean,ring_1_mean,ring_2_mean
0,0,/Users/renaud/datasets/synaptic_proteins/clust...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,1.000000,135.332001,0.925953,1.000000,0.902193,0.184229,0.164634,...,45.120640,0.620878,0.820528,0.762327,0.698873,23.507147,7.040886,0.192833,0.339275,0.519159
1,1,/Users/renaud/datasets/synaptic_proteins/clust...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,0.703072,111.154327,0.948557,0.703072,0.898478,0.122644,0.089146,...,42.653957,0.563892,1.014210,0.492432,0.727969,21.996159,6.498264,0.138741,0.245279,0.392226
2,2,/Users/renaud/datasets/synaptic_proteins/clust...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,1.000000,328.048767,0.890856,1.000000,0.937135,0.192712,0.171771,...,93.911560,0.561250,0.483667,0.626849,0.648299,52.357281,7.529780,0.297228,0.487086,0.674885
3,3,/Users/renaud/datasets/synaptic_proteins/clust...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,1.000000,91.012192,0.968693,1.000000,0.890132,0.260882,0.047307,...,39.942474,0.577176,-0.240557,0.990872,0.752089,18.541162,6.760745,0.293316,0.455584,0.776251
4,4,/Users/renaud/datasets/synaptic_proteins/clust...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,0.634812,115.497475,0.955617,0.634812,0.907527,0.117984,0.080476,...,46.893444,0.494237,1.278584,0.431649,0.837790,24.462677,6.739009,0.157294,0.242821,0.395006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,606,/Users/renaud/datasets/synaptic_proteins/clust...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,0.489426,72.426407,0.684823,0.489426,0.895311,0.082255,0.069472,...,26.079990,0.289133,-0.984288,0.325171,0.970000,22.226492,6.641793,0.143977,0.152937,0.250069
607,607,/Users/renaud/datasets/synaptic_proteins/clust...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,1.000000,119.669044,0.859276,1.000000,0.944953,0.209245,0.077509,...,47.083599,0.249362,-0.251274,0.884075,0.965859,33.416031,8.071411,0.449750,0.451917,0.742148
608,608,/Users/renaud/datasets/synaptic_proteins/clust...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,1.000000,92.497475,0.779905,1.000000,0.935577,0.214977,0.090355,...,33.082180,0.379459,-0.715790,0.918172,0.902098,25.631847,7.899512,0.402035,0.456528,0.705635
609,609,/Users/renaud/datasets/synaptic_proteins/clust...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,0.208459,65.941124,0.378156,0.208459,0.780238,0.035019,0.020361,...,20.405333,0.272820,-1.375648,0.171198,0.936508,19.380548,5.695748,0.094507,0.102011,0.141102


In [ ]:
# Process with DINO-v2 model
custom_transformations = stedfm_stretch_transforms
checkpoint_path = "checkpoints/segmentation_dataset_clusters/clusters_stretch_dist8_area128_dinov2.h5"

pheno.process_images(wrapper, 
                    force_rgb=True,
                    channel_mode='combined',
                    checkpoint_path=checkpoint_path,
                    custom_transformations=custom_transformations,
                    )
pheno.compute_properties(property_preset="complete", checkpoint_path=checkpoint_path)

Processing batches:   0%|          | 0/46 [00:00<?, ?it/s]

Pipeline ready: 1449 images available (lazy loading from checkpoints/segmentation_dataset_clusters/clusters_rc_dinov2.h5).
Masks: 1449/1449 images.


Computing properties:   0%|          | 0/1449 [00:00<?, ?img/s]

Checkpoint: saved 1449 properties (refreshed).
NaNs detected in property columns (NaN count per column): {'ring_1_mean': 2, 'ring_1_std': 2}. 2/1449 images have at least one such NaN. Use detect_nan_properties() for per-column row indices.


,image_index,image_path,image_name,intensity_max,perimeter,eccentricity,intensity_max_masked,texture_correlation,intensity_std_masked,ring_3_std,...,major_axis_length,texture_homogeneity,orientation,ring_3_mean,solidity,equivalent_diameter,intensity_entropy,intensity_mean,ring_1_mean,ring_2_mean
0,0,/Users/renaud/datasets/synaptic_proteins/clust...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,1.000000,135.332001,0.925953,1.000000,0.902193,0.184229,0.164634,...,45.120640,0.620878,0.820528,0.762327,0.698873,23.507147,7.040886,0.192833,0.339275,0.519159
1,1,/Users/renaud/datasets/synaptic_proteins/clust...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,0.703072,111.154327,0.948557,0.703072,0.898478,0.122644,0.089146,...,42.653957,0.563892,1.014210,0.492432,0.727969,21.996159,6.498264,0.138741,0.245279,0.392226
2,2,/Users/renaud/datasets/synaptic_proteins/clust...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,1.000000,328.048767,0.890856,1.000000,0.937135,0.192712,0.171771,...,93.911560,0.561250,0.483667,0.626849,0.648299,52.357281,7.529780,0.297228,0.487086,0.674885
3,3,/Users/renaud/datasets/synaptic_proteins/clust...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,1.000000,71.012192,0.943404,1.000000,0.911773,0.213408,0.108634,...,31.900108,0.475545,-0.927284,0.796850,0.880282,17.841242,6.975996,0.235675,0.331333,0.570802
4,4,/Users/renaud/datasets/synaptic_proteins/clust...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,1.000000,91.012192,0.968693,1.000000,0.890132,0.260882,0.047307,...,39.942474,0.577176,-0.240557,0.990872,0.752089,18.541162,6.760745,0.293316,0.455584,0.776251
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1444,1444,/Users/renaud/datasets/synaptic_proteins/clust...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,1.000000,92.497475,0.779905,1.000000,0.935577,0.214977,0.090355,...,33.082180,0.379459,-0.715790,0.918172,0.902098,25.631847,7.899512,0.402035,0.456528,0.705635
1445,1445,/Users/renaud/datasets/synaptic_proteins/clust...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,0.208459,65.941124,0.378156,0.208459,0.780238,0.035019,0.020361,...,20.405333,0.272820,-1.375648,0.171198,0.936508,19.380548,5.695748,0.094507,0.102011,0.141102
1446,1446,/Users/renaud/datasets/synaptic_proteins/clust...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,0.184290,48.284271,0.473148,0.184290,0.772092,0.032093,0.022309,...,16.331713,0.287986,0.379809,0.130111,0.958333,15.306080,5.227383,0.054653,0.062722,0.096092
1447,1447,/Users/renaud/datasets/synaptic_proteins/clust...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,0.075529,37.556351,0.697358,0.075529,0.544121,0.012628,0.014110,...,13.527584,0.344171,-0.917750,0.050604,0.909910,11.340071,4.106659,0.031427,0.037405,0.041031


In [8]:
# Plot number of images per group
pheno.plot_counts(group_by=["protein"])

## 3. Summaries by Experimental Group

This step checks if your physical measurements (like overall area or brightness) actually differ between your experimental groups (like 'Treated' vs. 'Untreated'). It requires that you have already run `compute_properties`.

In [19]:
prop_keys = pheno.get_available_property_keys()

df_grouped = pheno.property_stats_by_group(
    group_by=["protein"],
    print_properties=prop_keys[:3],  # first 3 properties for brevity
)
# Grouped statistics table is printed when print_table=True (default).


                      PROPERTY STATISTICS BY PROTEIN                      
Protein  N  |     Area (μ±σ)     |  Circula... (μ±σ)  |  Eccentr... (μ±σ) 
--------------------------------------------------------------------------
 Homer  266 |   261.98±330.06    |     0.69±0.25      |     0.79±0.17     
 PSD95  291 |   326.36±322.06    |     0.73±0.23      |     0.75±0.18     
  Rim   453 |   264.02±201.87    |     0.62±0.28      |     0.78±0.15     
Bassoon 439 |   355.22±265.59    |     0.80±0.19      |     0.70±0.17     


### 3.1.1 Show top features different from reference

Shows which morphological features differ most between each group and a reference (e.g. control). This uses **Cohen's d** (effect size) by default to rank properties by how significant the difference is, regardless of the property's units.

*   **Cohen's d > 0.8** is generally considered a "large" effect.
*   **Rank by |d|** ensures we find the features that change most in either direction (larger or smaller).

Adjust `reference_group` to match your metadata.

In [ ]:
pheno.top_properties_different_from_reference(
    df_grouped,
    reference_group={"synapse_class": "round"},
    k=3,
)

## 4. Cluster in embedding space

Clusters are **data-driven** groups in embedding space; they may or may not match your experimental labels. Set `n_clusters` between 2 and the number of images (see [Pipeline API](../../docs/src/content/docs/advanced/api/pipeline.md)). After clustering, labels are stored under metadata key `cluster`.

In [ ]:
pheno.compute_clustering(n_clusters=4, clustering_method="kmeans")
print(pheno.get_available_property_keys())
print(pheno.get_available_metadata_keys())

pheno.plot_tsne(color_by="synapse_class")

## 8. Interactive Exploration

The pipeline provides an interactive dashboard using Plotly and ipywidgets. This allows you to explore the embedding space, highligh specific groups, and inspect individual images by clicking on dots.

Note: This requires a Jupyter environment with `ipywidgets` and `plotly` installed.


In [61]:
# load STED-FM results
# clusters_rc_dist8_area128_
checkpoint_path = "checkpoints/segmentation_dataset_clusters/clusters_stretch_dist8_area128_stedfm.h5"
#checkpoint_path = "checkpoints/segmentation_dataset_clusters/clusters_pad_stedfm.h5"
pheno.load_results(checkpoint_path)

Pipeline ready: 1084 images available (lazy loading from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/segmentation_dataset_clusters/clusters_stretch_dist8_area128_stedfm.h5).
Loaded 1084 images from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/segmentation_dataset_clusters/clusters_stretch_dist8_area128_stedfm.h5 (hdf5), 384-dim embeddings (lazy, on disk).
Content: 32 metadata keys (antibody_label, area, bbox_col_max, bbox_col_min, bbox_padding...), 32 properties (area, circularity, eccentricity, equivalent_diameter, euler_number...).


In [ ]:
# load DinoV2 results
checkpoint_path = "checkpoints/segmentation_dataset_clusters/clusters_rp_dinov2.h5"
pheno.load_results(checkpoint_path)

Pipeline ready: 1449 images available (lazy loading from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/segmentation_dataset_clusters/clusters_rp_dinov2.h5).
NaNs detected in property columns (NaN count per column): {'ring_1_mean': 2, 'ring_1_std': 2}. 2/1449 images have at least one such NaN. Use detect_nan_properties() for per-column row indices.
Loaded 1449 images from /Users/renaud/python-projects/phenome-project/PhenoMe/Notebooks/sted-fm/checkpoints/segmentation_dataset_clusters/clusters_rp_dinov2.h5 (hdf5), 384-dim embeddings (lazy, on disk).
Content: 32 metadata keys (antibody_label, area, bbox_col_max, bbox_col_min, bbox_padding...), 32 properties (area, circularity, eccentricity, equivalent_diameter, euler_number...).


In [ ]:
outliers = pheno.detect_outliers(
    group_by="synapse_class", # We expect cells within the same condition to look similar
    threshold=3.0         # Z-score distance threshold
)

if outliers["outlier_indices"]:
    print(f"Found {len(outliers['outlier_indices'])} outliers across all groups.")
    top_idx = outliers["outlier_indices"][0]
    print(f"Plotting the most extreme outlier (index: {top_idx})...")
    pheno.plot_image_by_index(top_idx, show_extra_info=False)
else:
    print("No outliers detected with current settings.")

In [75]:
# Launch interactive explorer (if in Jupyter)
explorer = pheno.create_interactive_explorer()